# HDD 13 — Broader model/event coverage audit and prospective Q4 reservation

This notebook reuses the **same Q3 ZIP**. It ranks provisional model candidates using July 1 population and feature availability only, freezes a fresh drive split, then audits Q3 development coverage and recorded failures. Q4 observations are not read. The prospective test period is October 1–December 31, 2021. The original June benchmark stays untouched.

The proposal requires at least five eligible HDD models, multiple seeds, stronger baselines and drive-level uncertainty estimates. This notebook addresses data feasibility and reservation; it does not complete those experiments. Candidate device types other than the known pilot model must be verified from manufacturer specifications before the main study. Capacity and SMART availability do not prove a device is an HDD.

**Workload:** the archive is streamed twice: first without labels for quality coverage, then with only identifiers and failure flags for the development event audit. No full-fleet daily tensor is kept in memory. Let each pass finish before running the following cell.

In [1]:
from pathlib import Path
import json,hashlib,zipfile,re
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
OUT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
EVAL_OUT=COMPONENT_ROOT/'data/evaluation_only'/RUN_ID/AUDIT_ID
SPLIT_PATH=COMPONENT_ROOT/'data/splits'/f'{AUDIT_ID}_serials.json'
for p in [OUT,EVAL_OUT,SPLIT_PATH.parent]:p.mkdir(parents=True,exist_ok=True)
INPUTS={}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def remember(p):
    assert p.is_file(),f'Missing input: {p}'
    key=str(p.relative_to(COMPONENT_ROOT)) if p.is_relative_to(COMPONENT_ROOT) else str(p)
    INPUTS[key]=sha256(p);return p
def load_json(p):return json.loads(remember(p).read_text())
up=COMPONENT_ROOT/'results'/RUN_ID/'q3_recorded_failure_evaluation_v1/recorded_failure_evaluation_complete.json'
previous=load_json(up);assert previous['analysis_complete'] and not previous['sealed_June_test_opened']
exp=load_json(COMPONENT_ROOT/'results'/RUN_ID/'q3_development_expansion_v1/expansion_complete.json')
assert Q3_ZIP.is_file(),f'Use the Q3 ZIP from Notebook 09: {Q3_ZIP}'
assert sha256(remember(Q3_ZIP))==exp['settings']['archive_sha256']
old=load_json(COMPONENT_ROOT/'data/splits'/f'{RUN_ID}_serials.json')
q3old=load_json(COMPONENT_ROOT/'results'/RUN_ID/'q3_development_expansion_v1/q3_serials.json')
assert all(isinstance(v,list) and all(isinstance(s,str) for s in v) for v in old.values())
assert all(isinstance(v,list) and all(isinstance(s,str) for s in v) for v in q3old.values())
EXCLUDED=set().union(*(set(v) for v in list(old.values())+list(q3old.values())))
FEATURES=['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
SETTINGS={'cohort_date':'2021-07-01','minimum_fresh_first_day_drives':500,
    'minimum_complete_feature_fraction_first_day':.90,'minimum_capacity_bytes':1000000000000,
    'maximum_provisional_models':5,'test_drive_fraction':.20,'split_salt':'multimodel-q4-reservation-v1-seed42',
    'window_days':30,'warning_horizons_days':[7,14,30],
    'test_endpoint_start':'2021-10-01','test_endpoint_end':'2021-12-31',
    'model_selection_rule':'rank eligible July 1 fresh populations by drive count; break ties by model name; no failure counts used',
    'archive_sha256':sha256(Q3_ZIP),'upstream_evaluation_sha256':sha256(up),
    'excluded_serials_sha256':hashlib.sha256(json.dumps(sorted(EXCLUDED)).encode()).hexdigest()}
p=OUT/'audit_settings.json'
if p.exists():assert json.loads(p.read_text())==SETTINGS,'Changed audit protocol/input: use a new AUDIT_ID and preserve the old reservation.'
p.write_text(json.dumps(SETTINGS,indent=2))
with zipfile.ZipFile(Q3_ZIP) as z:
    MEMBERS={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and '2021-07-01'<=match.group(1)<='2021-09-30':
            date=match.group(1);assert date not in MEMBERS,'Duplicate dated member.'
            MEMBERS[date]=info.filename
    DATES=pd.date_range('2021-07-01','2021-09-30')
    assert set(MEMBERS)==set(DATES.strftime('%Y-%m-%d')),'Missing Q3 daily files.'
print('Excluded previously reserved/used serials:',len(EXCLUDED))

Excluded previously reserved/used serials: 1450


## 1. Select provisional model candidates before inspecting failure labels

No individual drive is filtered by its SMART values or eventual survival. A model is provisionally eligible if at least 500 fresh drives are present on July 1, at least 90% of that model's fresh rows have all five valid features, and its median reported capacity is at least 1 TB. This is a **declared pilot feasibility rule**, not a statistical power calculation. Ranking is fixed before failure extraction.

Every selected model still needs hardware-type confirmation. If fewer than five candidates pass, the audit continues with the available models and records the shortfall. Do not relax thresholds after inspecting failures under the same experiment ID.

In [2]:
def numeric_valid(frame):
    values=frame[FEATURES].apply(pd.to_numeric,errors='raise').to_numpy(dtype=float)
    return np.isfinite(values)&(values>=0)
with zipfile.ZipFile(Q3_ZIP) as z:
    with z.open(MEMBERS['2021-07-01']) as f:
        cohort=pd.read_csv(f,usecols=['serial_number','model','capacity_bytes']+FEATURES,
            dtype={'serial_number':'string','model':'string'})
assert cohort.serial_number.notna().all() and cohort.model.notna().all()
assert not cohort.serial_number.duplicated().any(),'Duplicate July 1 serial.'
cohort=cohort.loc[~cohort.serial_number.isin(EXCLUDED)].copy()
cohort['complete_features']=numeric_valid(cohort).all(1)
cohort['capacity_bytes']=pd.to_numeric(cohort.capacity_bytes,errors='raise')
model_audit=cohort.groupby('model',sort=True).agg(fresh_first_day_drives=('serial_number','size'),
    complete_feature_fraction=('complete_features','mean'),median_capacity_bytes=('capacity_bytes','median')).reset_index()
model_audit['provisionally_eligible']=(model_audit.fresh_first_day_drives.ge(SETTINGS['minimum_fresh_first_day_drives']) &
    model_audit.complete_feature_fraction.ge(SETTINGS['minimum_complete_feature_fraction_first_day']) &
    model_audit.median_capacity_bytes.ge(SETTINGS['minimum_capacity_bytes']))
ranked=model_audit.loc[model_audit.provisionally_eligible].sort_values(['fresh_first_day_drives','model'],ascending=[False,True])
MODELS=ranked.head(SETTINGS['maximum_provisional_models']).model.tolist()
assert MODELS,'No eligible model population. Preserve this result and review source coverage before further preparation.'
model_audit['selected_for_audit']=model_audit.model.isin(MODELS)
model_audit['device_type_verified']=model_audit.model.eq('ST12000NM0008')
model_audit.to_csv(OUT/'first_day_model_eligibility.csv',index=False)
display(model_audit.loc[model_audit.selected_for_audit])
print('Provisional model count:',len(MODELS),'| at least five available:',len(MODELS)>=5)

,model,fresh_first_day_drives,complete_feature_fraction,median_capacity_bytes,provisionally_eligible,selected_for_audit,device_type_verified
4,HGST HMS5C4040BLE640,12656,0.994232,4.000787e+12,True,True,False
18,ST12000NM0008,18882,1.000000,1.200014e+13,True,True,True
27,ST4000DM000,18792,1.000000,4.000787e+12,True,True,False
39,ST8000NM0055,14401,1.000000,8.001563e+12,True,True,False
48,TOSHIBA MG07ACA14TA,32032,0.999126,1.400052e+13,True,True,False


Provisional model count: 5 | at least five available: True


## 2. Reserve fresh future-test drives before any event audit

Within each selected July 1 model population, rank serials by a fixed salted hash and reserve the first 20% for the prospective Q4 test. No future-survival filter is applied. The remaining drives form the Q3 audit/development pool. Keep absent test drives in the declaration; absence and attrition must later be reported.

Q3 records from the newly reserved test drives are excluded from both detailed audit passes below. Their Q4 labels remain unopened. September context for their eventual October windows must be loaded only after freezing the final modelling protocol. This is a reservation, not a ready-to-run final-test experiment or a guarantee of adequate Q4 events.

In [3]:
def split_rank(serial):return hashlib.sha256((SETTINGS['split_salt']+'|'+serial).encode()).hexdigest()
RESERVATION={};POOL_IDS=set();FUTURE_TEST_IDS=set()
for model in MODELS:
    serials=sorted(cohort.loc[cohort.model.eq(model),'serial_number'].tolist(),key=split_rank)
    n_test=max(1,int(np.floor(len(serials)*SETTINGS['test_drive_fraction'])))
    holdout=serials[:n_test];development=serials[n_test:]
    RESERVATION[model]={'development':development,'future_test':holdout}
    POOL_IDS.update(development);FUTURE_TEST_IDS.update(holdout)
assert POOL_IDS.isdisjoint(FUTURE_TEST_IDS)
assert (POOL_IDS|FUTURE_TEST_IDS).isdisjoint(EXCLUDED)
reservation={'settings_sha256':sha256(OUT/'audit_settings.json'),'models':RESERVATION,
    'test_endpoint_start':SETTINGS['test_endpoint_start'],'test_endpoint_end':SETTINGS['test_endpoint_end'],
    'future_test_outcomes_opened':False,'selection_uses_failure_labels':False}
if SPLIT_PATH.exists():assert json.loads(SPLIT_PATH.read_text())==reservation,'Reservation changed: preserve the original and investigate.'
SPLIT_PATH.write_text(json.dumps(reservation,indent=2))
SPLIT_HASH=sha256(SPLIT_PATH)
# Compact positions allow a small state per drive instead of loading all daily rows.
pool=cohort.loc[cohort.serial_number.isin(POOL_IDS),['serial_number','model']].sort_values(['model','serial_number']).reset_index(drop=True)
serial_index=pd.Index(pool.serial_number)
model_index={m:i for i,m in enumerate(MODELS)}
model_codes=pool.model.map(model_index).to_numpy(dtype=int)
N=len(pool);M=len(MODELS)
last_observed=np.full(N,-2,dtype=np.int16);valid_streak=np.zeros(N,dtype=np.int16)
window_bits=[0]*N
print('Declared Q3 audit/development drives:',len(POOL_IDS))
print('Reserved fresh future-test drives:',len(FUTURE_TEST_IDS))
print('No Q4 observations or failure labels have been opened.')

Declared Q3 audit/development drives: 77412
Reserved fresh future-test drives: 19351
No Q4 observations or failure labels have been opened.


## 3. Audit Q3 feature coverage without reading failure flags

A valid raw window needs 30 consecutive observed days with all five finite, nonnegative features. A missing day resets the streak. For each drive, a compact bit mask records dates with valid 30-day endpoints; it is later joined with evaluation-only failure events to count usable warning histories.

The audit retains the declared July 1 population regardless of attrition. Quality windows are counted before knowing failures and may include failure-day or later records; the event-support calculation only uses endpoints strictly preceding the first failure.

In [4]:
coverage=[]
with zipfile.ZipFile(Q3_ZIP) as z:
    for day,date in enumerate(DATES.strftime('%Y-%m-%d')):
        row_counts=np.zeros(M,dtype=np.int64);complete_counts=np.zeros(M,dtype=np.int64)
        window_counts=np.zeros(M,dtype=np.int64);feature_counts=np.zeros((M,len(FEATURES)),dtype=np.int64)
        seen=set()
        with z.open(MEMBERS[date]) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model']+FEATURES,
                    dtype={'serial_number':'string','model':'string'},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(POOL_IDS)].copy()
                if part.empty:continue
                ids=part.serial_number.tolist();assert len(ids)==len(set(ids)) and seen.isdisjoint(ids),'Duplicate drive-date.'
                seen.update(ids)
                assert pd.to_datetime(part.date,errors='raise').eq(pd.Timestamp(date)).all()
                pos=serial_index.get_indexer(part.serial_number);assert (pos>=0).all()
                assert part.model.tolist()==pool.iloc[pos].model.tolist(),'Serial changed model.'
                valid=numeric_valid(part);complete=valid.all(1);codes=model_codes[pos]
                row_counts+=np.bincount(codes,minlength=M)
                complete_counts+=np.bincount(codes[complete],minlength=M)
                for j in range(len(FEATURES)):feature_counts[:,j]+=np.bincount(codes[valid[:,j]],minlength=M)
                consecutive=last_observed[pos]==day-1
                valid_streak[pos]=np.where(complete,np.where(consecutive,valid_streak[pos]+1,1),0)
                last_observed[pos]=day
                accepted=pos[valid_streak[pos]>=SETTINGS['window_days']]
                window_counts+=np.bincount(model_codes[accepted],minlength=M)
                bit=1<<day
                for idx in accepted:window_bits[idx]|=bit
        for model,code in model_index.items():
            row={'date':date,'model':model,'declared_drives':len(RESERVATION[model]['development']),
                'observed_drive_records':int(row_counts[code]),'complete_feature_records':int(complete_counts[code]),
                'quality_30day_windows':int(window_counts[code])}
            for j,feature in enumerate(FEATURES):row[feature+'_valid_records']=int(feature_counts[code,j])
            coverage.append(row)
        if date.endswith(('01','15')):print('Quality pass:',date)
coverage=pd.DataFrame(coverage);coverage.to_csv(OUT/'daily_model_coverage.csv',index=False)
# Cache compact state for provenance and reproducible event-support computation.
state=pool.copy();state['last_observed_day_index']=last_observed
state['valid_window_endpoint_bits_hex']=[hex(v) for v in window_bits]
state.to_parquet(OUT/'development_window_support.parquet',index=False)
print('Quality pass complete. Failure labels have not yet been used in this audit.')

Quality pass: 2021-07-01
Quality pass: 2021-07-15
Quality pass: 2021-08-01
Quality pass: 2021-08-15
Quality pass: 2021-09-01
Quality pass: 2021-09-15
Quality pass complete. Failure labels have not yet been used in this audit.


## 4. Separately extract first Q3 failures for development drives only

Read only `date`, `serial_number` and `failure`. Failure counts do not change model selection, drive assignment or window acceptance. First events define evaluation support; no failure-enriched training sample is created. The newly reserved test drives remain excluded.

In [5]:
first_failure=np.full(N,-1,dtype=np.int16)
with zipfile.ZipFile(Q3_ZIP) as z:
    for day,date in enumerate(DATES.strftime('%Y-%m-%d')):
        seen=set()
        with z.open(MEMBERS[date]) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','failure'],dtype={'serial_number':'string'},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(POOL_IDS)].copy()
                if part.empty:continue
                ids=part.serial_number.tolist();assert len(ids)==len(set(ids)) and seen.isdisjoint(ids),'Duplicate evaluation drive-date.'
                seen.update(ids)
                assert pd.to_datetime(part.date,errors='raise').eq(pd.Timestamp(date)).all()
                flags=pd.to_numeric(part.failure,errors='raise')
                assert flags.notna().all() and flags.isin([0,1]).all(),'Missing/nonbinary failure flag.'
                pos=serial_index.get_indexer(part.serial_number);assert (pos>=0).all()
                failed=pos[flags.eq(1).to_numpy()]
                first_failure[failed]=np.where(first_failure[failed]<0,day,first_failure[failed])
        if date.endswith(('01','15')):print('Evaluation-only event pass:',date)
failed_positions=np.flatnonzero(first_failure>=0)
event_rows=[]
for idx in failed_positions:
    d=int(first_failure[idx]);row={'serial_number':pool.iloc[idx].serial_number,
        'model':pool.iloc[idx].model,'failure_date':DATES[d]}
    for h in SETTINGS['warning_horizons_days']:
        # h endpoint dates immediately before failure, each with a valid 30-day feature window.
        required=((1<<h)-1)<<(d-h) if d>=h else None
        row[f'complete_{h}day_warning_history']=bool(required is not None and (window_bits[idx]&required)==required)
    event_rows.append(row)
event_columns=['serial_number','model','failure_date']+[f'complete_{h}day_warning_history' for h in SETTINGS['warning_horizons_days']]
events=pd.DataFrame(event_rows,columns=event_columns)
events.to_csv(EVAL_OUT/'development_first_failure_support.csv',index=False)
assert set(events.serial_number).isdisjoint(FUTURE_TEST_IDS)
print('First Q3 development failure events:',len(events))

Evaluation-only event pass: 2021-07-01
Evaluation-only event pass: 2021-07-15
Evaluation-only event pass: 2021-08-01
Evaluation-only event pass: 2021-08-15
Evaluation-only event pass: 2021-09-01
Evaluation-only event pass: 2021-09-15
First Q3 development failure events: 230


## 5. Save population and event-support summaries

A complete warning history means features are available; it is not evidence that a detector would issue an alert. Few events yield imprecise recall estimates even with many overlapping windows. No minimum event count here is presented as a power guarantee, and no model is dropped because its failure count is inconvenient.

Next use the coverage results to define the larger study: confirm HDD device types, prepare chronological model-specific training/calibration data, run the declared baselines and seeds, then freeze the entire protocol before Q4 scoring. Do not open Q4 outcomes merely to choose a more favourable test sample.

In [6]:
summary=[]
for model in MODELS:
    sub=coverage.loc[coverage.model.eq(model)];ev=events.loc[events.model.eq(model)]
    positions=np.flatnonzero(model_codes==model_index[model])
    rows=int(sub.observed_drive_records.sum())
    row={'model':model,'declared_development_drives':len(RESERVATION[model]['development']),
        'reserved_future_test_drives':len(RESERVATION[model]['future_test']),
        'development_drives_ever_observed':int((last_observed[positions]>=0).sum()),
        'observed_drive_records':rows,'complete_feature_record_fraction':float(sub.complete_feature_records.sum()/rows) if rows else np.nan,
        'quality_30day_windows':int(sub.quality_30day_windows.sum()),'first_Q3_failure_events':len(ev),
        'device_type_verified':model=='ST12000NM0008'}
    for h in SETTINGS['warning_horizons_days']:row[f'failure_events_with_complete_{h}day_history']=int(ev[f'complete_{h}day_warning_history'].sum())
    summary.append(row)
summary=pd.DataFrame(summary);summary.to_csv(OUT/'model_event_coverage_summary.csv',index=False)
display(summary)
# Verify the bit-mask definition on exact, incomplete and boundary histories.
def complete_history(bits,day,h):
    if day<h:return False
    required=((1<<h)-1)<<(day-h)
    return (bits&required)==required
assert complete_history(sum(1<<d for d in range(29,40)),40,7)
assert not complete_history(sum(1<<d for d in range(29,40) if d!=35),40,7)
assert not complete_history(0,3,7)
# All cohort drives stay declared; no survivor-based sample replacement.
assert len(pool)==len(POOL_IDS) and POOL_IDS.isdisjoint(FUTURE_TEST_IDS)
assert sha256(SPLIT_PATH)==SPLIT_HASH
for location,digest in INPUTS.items():
    p=Path(location) if Path(location).is_absolute() else COMPONENT_ROOT/location
    assert sha256(p)==digest,f'Input changed: {p}'
outputs={str(p.relative_to(COMPONENT_ROOT)):sha256(p) for folder in [OUT,EVAL_OUT] for p in sorted(folder.iterdir()) if p.is_file() and p.name!='multimodel_coverage_complete.json'}
completion={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,'run_id':RUN_ID,
    'audit_id':AUDIT_ID,'settings':SETTINGS,'input_sha256':INPUTS,'output_sha256':outputs,
    'split_path':str(SPLIT_PATH.relative_to(COMPONENT_ROOT)),'split_sha256':SPLIT_HASH,
    'provisional_models':MODELS,'at_least_five_provisional_models':len(MODELS)>=5,
    'all_device_types_verified':bool(summary.device_type_verified.all()),
    'development_failure_events':len(events),'future_test_outcomes_opened':False,'sealed_June_test_opened':False,
    'failure_labels_used_for_event_audit_only':True,'failure_labels_used_for_model_or_drive_selection':False,
    'model_fits':0,'final_test_evaluated':False,
    'limitations':['Q3 development audit only','device-type confirmation pending for new model names',
        'first-day eligibility/ranking conditions the model population','Q3 initial history limits early warning coverage',
        'prospective Q4 event sufficiency unknown','earlier-than-Q3 failure history unavailable for new drives',
        'no main-study training, baseline/seed comparison or uncertainty estimates performed']}
(OUT/'multimodel_coverage_complete.json').write_text(json.dumps(completion,indent=2,allow_nan=False))
print('NOTEBOOK 13 COMPLETE — broader development event audit and future-test reservation saved.')
print('Provisional models:',len(MODELS),'| development events:',len(events))
print('Q4 outcomes unopened. Hardware types for new model candidates still require verification.')
# print('Share this executed notebook, model_event_coverage_summary.csv and first_day_model_eligibility.csv.')
print('Results:',OUT)

,model,declared_development_drives,reserved_future_test_drives,development_drives_ever_observed,observed_drive_records,complete_feature_record_fraction,quality_30day_windows,first_Q3_failure_events,device_type_verified,failure_events_with_complete_7day_history,failure_events_with_complete_14day_history,failure_events_with_complete_30day_history
0,TOSHIBA MG07ACA14TA,25626,6406,25626,2355314,0.999962,1611507,47,False,29,25,18
1,ST12000NM0008,15106,3776,15106,1387600,0.999924,945687,45,True,28,22,17
2,ST4000DM000,15034,3758,15034,1379794,0.999952,942384,71,False,43,40,27
3,ST8000NM0055,11521,2880,11521,1056841,0.999678,715698,61,False,37,30,16
4,HGST HMS5C4040BLE640,10125,2531,10125,930661,0.999583,633170,6,False,1,1,0


NOTEBOOK 13 COMPLETE — broader development event audit and future-test reservation saved.
Provisional models: 5 | development events: 230
Q4 outcomes unopened. Hardware types for new model candidates still require verification.
Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_multimodel_event_coverage_v1
